[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# prefetch and Load


## What you will be able to do

Say why the fix from the **Relationships** notebook does not work in the other direction, and what
does. Load a parent and its children in two queries with `with_related(Load(...))` or with
`prefetch`, and measure it rather than hoping. Read the three strategies a child query can use to
name its parents, and say which one stops working as the number of parents grows, and why. Keep only
the first few children per parent with `per_parent`, and filter the children with a query of your
own. Recognize the call that quietly returns no related rows at all.


## The idea

### The problem

The **Relationships** notebook fixed one query per row with `select(Book, Author).join(Author)`:
ask for both tables' columns and peewee builds both objects from one row. That works because each
book has exactly one author, so the pair fits on a row.

Go the other way and it does not fit. One author has three books, so a join gives three rows for one
author, and there is no single row to build an `Author` with its `books` from. `author.books` in a
loop is therefore one query per author, and the trick that fixed the other direction is not
available.

### What a prefetch is

Fetch the parents. Then fetch all of their children with one more query. Then put each child on its
parent in Python. Two queries for any number of parents, at the cost of peewee doing the matching
that a join would have done in the database.

peewee spells this two ways: `prefetch(parents, children)`, which has been there for years, and
`parents.with_related(Load(Parent.children))`, which is newer and takes options `prefetch` cannot.

### Why it works that way

A relational result is a rectangle. Anything that repeats, such as three books for one author, has
to be either repeated rows or a second query. Joining gives repeated rows, which is fine when you
want a flat listing and wasteful when you want objects, because the author's columns come back once
per book. Two queries gives each row once and asks Python to assemble them.

### Where this shows up

Every page that shows a list of things with some of each thing's children: posts with their
comments, orders with their lines, authors with their books. It is the same shape as the N plus one
from the **Relationships** notebook, and the same invisible cost.

### What this notebook covers

Why the join fix does not apply. Two queries instead of five, two ways, measured. The three
strategies the child query can use to name its parents, read from the SQL and the values each
carries. The strategy that works until there are too many parents, shown failing. `per_parent` and a
filtered child query, which are the reason `Load` exists. Then the four failures, one of them a call
that succeeds and returns nothing.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from peewee import CharField, ForeignKeyField, Load, Model, SqliteDatabase
from playhouse.test_utils import count_queries

db = SqliteDatabase(":memory:")


class Author(Model):
    name = CharField()

    class Meta:
        database = db


class Book(Model):
    title = CharField()
    author = ForeignKeyField(Author, backref="books")

    class Meta:
        database = db


db.create_tables([Author, Book])
for name in ("Ursula Vance", "Marco Pietra", "Ines O'Brien", "Kofi Mensah"):
    author = Author.create(name=name)
    for number in range(3):
        Book.create(title=f"{name} {number}", author=author)

with count_queries() as counter:
    for author in Author.select():                      # one query, then one for each author
        [book.title for book in author.books]
print("author.books in a loop:  ", counter.count, "queries")

with count_queries() as counter:
    for author in Author.select().with_related(Load(Author.books)):
        [book.title for book in author.books]
print("with_related(Load(...)): ", counter.count, "queries")
```

```
author.books in a loop:   5 queries
with_related(Load(...)):  2 queries
```

Five against two, and two is two whether there are four authors or forty thousand. The loop is
identical in both: what changed is that the books were already in hand by the time it ran.


## Setup

Six imports, peewee installed and pinned, the catalog's models, and a database that records.

- `peewee` is the library, and `Model`, the field classes and `SqliteDatabase`, from it, are what a
  model is written with
- `Load`, `prefetch` and `PREFETCH_TYPE`, also from peewee, are this notebook's subject, and
  `chunked` builds the large table one section needs
- `count_queries`, from `playhouse.test_utils`, counts the queries a block sends
- `subprocess`, `sys`, `version` and `PackageNotFoundError` install peewee 4.5.1 where the version is
  not that, as on Colab, whose 4.4.0 words some of these messages differently
- `AUTHORS` and `BOOKS` are the catalog, and `build` makes the tables and loads them

`db` is a `RecordingSqlite`, which keeps each statement and how many values it carried. The second
number is what this notebook is really about: the three strategies below send the same number of
queries and differ in how many values they put into one of them, and that difference is the whole
story of which one stops working first.


In [1]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import (PREFETCH_TYPE, CharField, ForeignKeyField, IntegerField, Load, Model,
                    OperationalError, SqliteDatabase, chunked, prefetch)
from playhouse.test_utils import count_queries

AUTHORS = [                                                         # name, the year of the first book
    ("Ursula Vance", 2014),
    ("Marco Pietra", 2009),
    ("Ines O'Brien", 1998),
    ("Kofi Mensah", 2015),
]

BOOKS = [                                                           # title, author, year, pages
    ("The Salt Road", "Ursula Vance", 2014, 312),
    ("Nightjar", "Ursula Vance", 2018, 244),
    ("The Quiet Engine", "Ursula Vance", 2021, 398),
    ("Stone and Tide", "Marco Pietra", 2009, 501),
    ("The Lantern Keeper", "Marco Pietra", 2016, 276),
    ("Riverwork", "Marco Pietra", 2022, 189),
    ("A Careful Fire", "Ines O'Brien", 1998, 420),
    ("The Long Field", "Ines O'Brien", 2004, 355),
    ("Winter Harbour", "Ines O'Brien", 2011, 263),
    ("The Drum Line", "Kofi Mensah", 2015, 198),
    ("Harmattan", "Kofi Mensah", 2019, 331),
    ("Small Machines", "Kofi Mensah", 2023, 287),
]

class RecordingSqlite(SqliteDatabase):
    """A database that keeps each statement sent through it, and how many values it carried."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.statements = []

    def execute_sql(self, sql, params=None):
        self.statements.append((" ".join(sql.split()), len(params or ())))
        return super().execute_sql(sql, params)

def bound(database, work):
    """Run some work and report the queries it sent and the values those queries carried."""
    database.statements.clear()
    work()
    return len(database.statements), sum(count for _, count in database.statements)

db = RecordingSqlite(":memory:", pragmas={"foreign_keys": 1})       # the guide's database line


class CatalogModel(Model):
    """Every model in the catalog names the database once, here."""

    class Meta:
        database = db


class Author(CatalogModel):
    name = CharField(max_length=60, unique=True)
    first_book = IntegerField()


class Book(CatalogModel):
    title = CharField(max_length=80)
    author = ForeignKeyField(Author, backref="books")
    year = IntegerField(index=True)
    pages = IntegerField()

def build(database):
    """Create the tables and load the catalog, in one transaction."""
    database.create_tables([Author, Book])
    with database.atomic():
        Author.insert_many([{"name": name, "first_book": year} for name, year in AUTHORS]).execute()
        written = {author.name: author.id for author in Author.select()}
        Book.insert_many([{"title": title, "author": written[author], "year": year, "pages": pages}
                          for title, author, year, pages in BOOKS]).execute()


build(db)
print("peewee", peewee.__version__, "|", Author.select().count(), "authors and",
      Book.select().count(), "books, three each")


peewee 4.5.1 | 4 authors and 12 books, three each


## Worked examples

### The direction a join cannot flatten

`select(Book, Author).join(Author)` works because a book has one author. Try it the other way and
count the rows:


In [2]:
joined = Author.select(Author, Book).join(Book)
print("authors:", Author.select().count(), "| rows the join returns:", len(joined))
print([(row.name, row.title) for row in joined.objects()][:4])


authors: 4 | rows the join returns: 12
[('Ursula Vance', 'The Salt Road'), ('Ursula Vance', 'Nightjar'), ('Ursula Vance', 'The Quiet Engine'), ('Marco Pietra', 'Stone and Tide')]


Twelve rows for four authors. Each author's name came back three times, and there is no single row
that holds an author together with all three of that author's books. A join can carry one parent
onto each child. It cannot carry many children onto one parent.

So `author.books` inside a loop goes back to the database each time:


In [3]:
with count_queries() as counter:
    for author in Author.select():
        [written.title for written in author.books]
print("one query for the authors, then one for each:", counter.count)


one query for the authors, then one for each: 5


### Two queries, two ways

`with_related` takes a `Load` node naming the relationship to follow:


In [4]:
with count_queries() as counter:
    loaded = list(Author.select().with_related(Load(Author.books)))
    for author in loaded:
        [written.title for written in author.books]
print("with_related(Load(Author.books)):", counter.count, "queries")

with count_queries() as counter:
    fetched = list(prefetch(Author.select(), Book.select()))
    for author in fetched:
        [written.title for written in author.books]
print("prefetch(Author.select(), Book.select()):", counter.count, "queries")


with_related(Load(Author.books)): 2 queries
prefetch(Author.select(), Book.select()): 2 queries


The same two queries either way, and `author.books` is now a list that is already in memory rather
than a query. Reading it costs nothing, which is the point:


In [5]:
author = loaded[0]
print("type of author.books after loading:", type(author.books).__name__)
with count_queries() as counter:
    [written.title for written in author.books]
print("reading it again:", counter.count, "queries")


type of author.books after loading: list
reading it again: 0 queries


### How the child query names its parents

Both queries fetch all the children at once, which means the second query has to say which parents
it wants children for. There are three ways to say it, and `PREFETCH_TYPE` names them:


In [6]:
for name in ("WHERE", "JOIN", "MATERIALIZE"):
    strategy = getattr(PREFETCH_TYPE, name)
    queries, values = bound(db, lambda: list(
        Author.select().with_related(Load(Author.books, strategy=strategy))))
    child = [line for line, _ in db.statements if '"book"' in line][0]
    print(f"  {name:<12} {queries} queries, {values:>2} values bound")
    print(f"               {child[child.index('WHERE') if 'WHERE' in child else child.index('INNER'):][:74]}")


  WHERE        2 queries,  0 values bound
               WHERE ("t1"."author_id" IN (SELECT "t2"."id" FROM "author" AS "t2"))
  JOIN         2 queries,  0 values bound
               INNER JOIN (SELECT DISTINCT "t2"."id" FROM "author" AS "t2") AS "t3" ON ("
  MATERIALIZE  2 queries,  4 values bound
               WHERE ("t1"."author_id" IN (?, ?, ?, ?))


`WHERE` puts the parent query inside the child query as a subquery. `JOIN` joins the child table
against the same parent query. Neither sends any parent keys, because the database works them out
again. `MATERIALIZE` takes the parent keys peewee already has and lists them, which is why it is the
only one carrying values.

Listing them is the fastest thing to ask a database when there are a few, and it is what breaks
first when there are many.

### The strategy that is right until it is not

Four authors is too small to show what grows. Here is a table big enough, built and used only in
this section:


In [7]:
class Crowd(Model):
    name = CharField(max_length=20)

    class Meta:
        database = db


class Item(Model):
    label = CharField(max_length=20)
    owner = ForeignKeyField(Crowd, backref="items")

    class Meta:
        database = db


db.create_tables([Crowd, Item])
with db.atomic():
    for piece in chunked([{"name": f"c{number}"} for number in range(33_000)], 500):
        Crowd.insert_many(piece).execute()
    for piece in chunked([{"label": f"i{number}", "owner": number + 1} for number in range(33_000)], 500):
        Item.insert_many(piece).execute()

print("owners:", Crowd.select().count())


owners: 33000


Now the number that grows. This is the values bound by the child query alone, under `MATERIALIZE`,
for a few sizes of parent list:


In [8]:
def child_values(work):
    """How many values the query against the item table carried."""
    db.statements.clear()
    work()
    return next(count for line, count in db.statements if '"item"' in line)


for parents in (10, 100, 1000):
    values = child_values(lambda: list(
        Crowd.select().limit(parents).with_related(
            Load(Crowd.items, strategy=PREFETCH_TYPE.MATERIALIZE))))
    print(f"  {parents:>5} parents -> {values} values bound in the child query")


     10 parents -> 10 values bound in the child query
    100 parents -> 100 values bound in the child query
   1000 parents -> 1000 values bound in the child query


One value per parent, exactly. The **Creating and Changing Rows** notebook met the limit on how many
values one statement may carry, and this is the same limit approached from the other side. With all
thirty three thousand owners it is reached:


In [9]:
for name in ("WHERE", "MATERIALIZE"):
    try:
        rows = list(Crowd.select().with_related(
            Load(Crowd.items, strategy=getattr(PREFETCH_TYPE, name))))
        print(f"  {name:<12} loaded {len(rows)} owners")
    except OperationalError as error:
        print(f"  {name:<12} peewee.OperationalError: {error}")


  WHERE        loaded 33000 owners
  MATERIALIZE  peewee.OperationalError: too many SQL variables


The same code, the same models, the same two queries, and one of them cannot run. The exact number
of parents where this happens depends on how the SQLite in front of you was built, which is the
point: a strategy chosen because it was quickest on the rows you had can stop working on rows you
have not seen yet. `WHERE`, the default, has no such ceiling, because it sends no keys at all.

### What Load can do that prefetch cannot

`per_parent` keeps only the first few children of each parent, which is how a page shows three
comments per post without fetching every comment:


In [10]:
for author in Author.select().order_by(Author.name).with_related(
        Load(Author.books, per_parent=2)):
    print(f"  {author.name:<15} {[written.title for written in author.books]}")


  Ines O'Brien    ['A Careful Fire', 'The Long Field']
  Kofi Mensah     ['The Drum Line', 'Harmattan']
  Marco Pietra    ['Stone and Tide', 'The Lantern Keeper']
  Ursula Vance    ['The Salt Road', 'Nightjar']


A query of your own filters the children, and the parents with no matching child come back with an
empty list rather than being dropped:


In [11]:
recent = Book.select().where(Book.year >= 2015).order_by(Book.year)

for author in Author.select().order_by(Author.name).with_related(Load(Author.books, recent)):
    print(f"  {author.name:<15} {[written.title for written in author.books]}")


  Ines O'Brien    []
  Kofi Mensah     ['The Drum Line', 'Harmattan', 'Small Machines']
  Marco Pietra    ['The Lantern Keeper', 'Riverwork']
  Ursula Vance    ['Nightjar', 'The Quiet Engine']


Both of those are the reason `Load` was added. `prefetch` takes the queries and nothing else, so
"the first two per parent" is not something it can express.

### When to reach for which

| What you want | How to write it |
|---|---|
| one parent for each of many children | `.select(Child, Parent).join(Parent)`, from **Relationships** |
| many children for each of many parents | `.with_related(Load(Parent.children))` |
| the same, in the older spelling | `prefetch(parents, children)` |
| only the first few children per parent | `Load(Parent.children, per_parent=n)` |
| only some of the children | `Load(Parent.children, Child.select().where(...))` |
| the child query not to list the parent keys | the default strategy, `PREFETCH_TYPE.WHERE` |
| the parent keys listed, on a small result | `strategy=PREFETCH_TYPE.MATERIALIZE` |

`with_related(Load(...))` is the default for the parent to children direction. Reach for `prefetch`
in code that already uses it, and for a strategy other than the default only when you have measured
something, and only where the number of parents is bounded.

### A page of authors, finished

A listing that shows each author with their two most recent books, in two queries, with the count
and the shape both printed.


In [12]:
def author_page(most_recent=2, since=2000):
    """Each author with their most recent books, in two queries however many authors there are."""
    children = Book.select().where(Book.year >= since).order_by(Book.year.desc())
    return list(Author.select()
                      .order_by(Author.name)
                      .with_related(Load(Author.books, children, per_parent=most_recent)))


with count_queries() as counter:
    page = author_page()
    lines = [f"{author.name}: {[(b.title, b.year) for b in author.books]}" for author in page]

for line in lines:
    print("  ", line)
print("  queries sent:", counter.count)


   Ines O'Brien: [('Winter Harbour', 2011), ('The Long Field', 2004)]
   Kofi Mensah: [('Small Machines', 2023), ('Harmattan', 2019)]
   Marco Pietra: [('Riverwork', 2022), ('The Lantern Keeper', 2016)]
   Ursula Vance: [('The Quiet Engine', 2021), ('Nightjar', 2018)]
  queries sent: 2


Two queries for the whole page. `per_parent` is doing real work here: the child query is ordered
newest first, so the two kept per author are the two most recent, and the books nobody is going to
show were never fetched.

### Where each part came from

| In the page | What it relies on | The section that showed it |
|---|---|---|
| `.with_related(Load(Author.books, ...))` | parents and children in two queries | Two queries, two ways |
| `per_parent=most_recent` | only the children the page will show | What Load can do |
| `Book.select().where(...).order_by(...)` | a child query of your own | What Load can do |
| the default strategy | a child query that binds no parent keys | How the child query names its parents |
| `count_queries` | a cost that is printed rather than assumed | **Relationships** |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/07-prefetch-and-load-solutions.ipynb).

**1.** Count the queries a loop over authors takes when it reads `author.books`, then do the same
with `with_related` and print both numbers.


In [13]:
# your code here


**2.** Do the same with `prefetch` instead, and show that `author.books` is a list afterwards rather
than a query.


In [14]:
# your code here


**3.** Print each author with only their longest book, using `per_parent` and an ordered child
query.


In [15]:
# your code here


**4.** Load each author with only their books from before 2010, and show which author comes back
with an empty list.


In [16]:
# your code here


**5.** Print how many values the child query binds under each of the three strategies, and say which
one grows with the number of parents.


In [17]:
# your code here


**6.** Show that adding `.dicts()` to the parent query leaves the related books out, with no error.


In [18]:
# your code here


## Common errors

### AttributeError: 'ModelSelect' object has no attribute 'load'


In [19]:
Author.select().load(Load(Author.books))


AttributeError: 'ModelSelect' object has no attribute 'load'

The node is called `Load` and the method is called `with_related`, which is an easy pair to mix up.
The method says what it does to the query, and the node says what to load.


In [20]:
print("the right spelling:", len(list(Author.select().with_related(Load(Author.books)))), "authors")


the right spelling: 4 authors


### ValueError: Load() expects a foreign-key or backref reference, got <CharField: Author.name>.


In [21]:
Load(Author.name)


ValueError: Load() expects a foreign-key or backref reference, got <CharField: Author.name>.

`Load` takes a relationship, not a column. `Author.books` is the `backref` from the `ForeignKeyField`
on `Book`, and `Book.author` is the field itself. Both are relationships and either can be loaded.
A plain column is not something there is anything to fetch for.


In [22]:
print("backref  ->", len(list(Author.select().with_related(Load(Author.books)))), "authors")
print("the field ->", len(list(Book.select().with_related(Load(Book.author)))), "books")


backref  -> 4 authors
the field -> 12 books


### No error, and no books on any author: dicts with with_related


In [23]:
rows = list(Author.select().dicts().with_related(Load(Author.books)))
print("one row:", rows[0])
print("is there a books key:", "books" in rows[0])

other_order = list(Author.select().with_related(Load(Author.books)).dicts())
print("the other order:", other_order[0])


one row: {'id': 1, 'name': 'Ursula Vance', 'first_book': 2014}
is there a books key: False
the other order: {'id': 1, 'name': 'Ursula Vance', 'first_book': 2014}


Both orders run, both return rows, and neither has the related books anywhere in them. `dicts()` says
the results are plain dictionaries, and a dictionary is not something peewee can attach children to,
so the loading is quietly dropped. Nothing raises because nothing is wrong: you asked for
dictionaries and you got dictionaries.

This is the one to watch for, because `.dicts()` is usually added later, as a speed-up, by somebody
who is not thinking about the `Load` further along the line. The same mistake inside the child query
does raise, and so does the `prefetch` version, which is worth seeing so the silent one stands out:


In [24]:
for label, attempt in (
        ("Load(Author.books, Book.select().dicts())",
         lambda: list(Author.select().with_related(Load(Author.books, Book.select().dicts())))),
        ("prefetch(Author.select(), Book.select().dicts())",
         lambda: list(prefetch(Author.select(), Book.select().dicts())))):
    try:
        attempt()
        print(f"  {label}\n     no error")
    except (ValueError, AttributeError) as error:
        print(f"  {label}\n     {type(error).__name__}: {error}")


  Load(Author.books, Book.select().dicts())
     ValueError: Load() query must return model instances, not dicts/tuples/namedtuples.
  prefetch(Author.select(), Book.select().dicts())
     AttributeError: 'dict' object has no attribute '__data__'


### ValueError: PREFETCH_TYPE.MATERIALIZE is not supported by prefetch(). Use Model.select().with_related() with a Load() node instead.


In [25]:
prefetch(Author.select(), Book.select(), prefetch_type=PREFETCH_TYPE.MATERIALIZE)


ValueError: PREFETCH_TYPE.MATERIALIZE is not supported by prefetch(). Use Model.select().with_related() with a Load() node instead.

`prefetch` supports the two strategies that send no parent keys, and the message names the call that
supports the third. It is worth reading as a statement about which of the two functions is the
current one: the options live on `Load`, and `prefetch` is kept for the code that already uses it.


In [26]:
loaded = list(Author.select().with_related(
    Load(Author.books, strategy=PREFETCH_TYPE.MATERIALIZE)))
print("with_related takes it:", [author.name for author in loaded][:2], "...")


with_related takes it: ['Ursula Vance', 'Marco Pietra'] ...


## Recap

- A join carries one parent onto each child and cannot carry many children onto one parent, so the
  fix from **Relationships** does not work in this direction.
- `parents.with_related(Load(Parent.children))` and `prefetch(parents, children)` both fetch the
  lot in two queries, and afterwards the children are a list rather than a query.
- The child query has to say which parents it wants. `WHERE` and `JOIN` send the parent query again
  and bind no keys. `MATERIALIZE` lists the keys peewee already has.
- Listing the keys binds one value per parent, so it meets the driver's limit on values in one
  statement, and stops working at a number of parents that depends on the build.
- `per_parent` keeps the first few children of each parent, and a child query of your own filters
  them. Neither is something `prefetch` can express, which is why `Load` exists.
- Adding `.dicts()` to the parent query drops the loading silently, in either order. Inside the
  child query it raises instead.


## What is next

The **JSON Columns** notebook puts a whole document in one column: `JSONField`, reaching into it
with a path, and the filter that looks numeric and compares text until `as_int()` is added to it.


---

&#8592; **Previous:** [Relationships](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/06-relationships.ipynb)  &nbsp;·&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [JSON Columns](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/08-json-columns.ipynb) &#8594;
